# Lab 04 — Gradient Descent
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · Week 4 lab · about 75 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/labs/lab04-gradient-descent/Lab04_Gradient_Descent.ipynb)

In Lab 03 you found the best line by computing the cost at every point of a grid. That only works with two numbers to choose. Today you write **gradient descent**: start somewhere, feel which way is downhill, take a step, repeat. It is the same recipe that trains neural networks and large language models. Then you use **scikit-learn** to fit a model with all 8 concrete inputs and test it on mixes it has never seen. Questions marked 🏠 may be finished after lab; they are still part of the lab.

### What you will be able to do
1. Compute the gradients ∂J/∂w and ∂J/∂b for linear regression and check them numerically.
2. Implement gradient descent with a simultaneous update and draw its path on the cost contour.
3. Choose a learning rate by reading learning curves (too small, good, too large, diverging).
4. Explain why feature scaling speeds up gradient descent.
5. Fit a multiple linear regression with scikit-learn, evaluate it on a test set with MAE, RMSE, and R², and interpret its coefficients with care.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_strength.csv` — 1,030 laboratory concrete tests (Yeh 1998, UCI, CC BY 4.0).

**AI-use label:** **AI-Assisted** — you may ask an AI tool for explanations or debugging help. Fill in the AI-use acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part 1 — Recap: data, model, and cost (5 min)
Same data and the same two functions you wrote in Lab 03 — just run this cell.

In [ ]:
df = pd.read_csv(get_data("concrete_strength.csv"))
df["wc_ratio"] = df["water_kg_m3"] / df["cement_kg_m3"]
x = df["wc_ratio"].to_numpy()       # feature: water-cement ratio (–)
y = df["strength_mpa"].to_numpy()   # target: compressive strength (MPa)
m = len(x)

def predict(x, w, b):
    """Predicted strength (MPa): f(x) = w*x + b."""
    return w * x + b

def compute_cost(x, y, w, b):
    """Cost J(w, b) = (1/2m) * sum of squared errors, in MPa²."""
    errors = predict(x, w, b) - y
    return np.sum(errors**2) / (2 * len(x))

w_best, b_best = np.polyfit(x, y, 1)   # the exact answer, for comparison later
print(f"m = {m} tests")
print(f"Cost at w = 0, b = 0:      J = {compute_cost(x, y, 0, 0):.1f} MPa²")
print(f"Best line (np.polyfit):    w = {w_best:.2f} MPa, b = {b_best:.2f} MPa, J = {compute_cost(x, y, w_best, b_best):.2f} MPa²")

## Part 2 — The gradient: which way is downhill? (15 min)
Imagine standing on the cost surface from Lab 03 in thick fog. You cannot see the bottom, but you can feel the slope
under your feet. Step downhill, feel again, step again.

The slope of J in the w direction (with b held fixed) is the **partial derivative** ∂J/∂w; the slope in the b
direction is ∂J/∂b. For our cost function they are:

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)x^{(i)}
\qquad\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)$$

In words:
- **∂J/∂b is the average error.** If the line predicts too high on average, it is positive → lower b.
- **∂J/∂w is the average of (error × x).** Errors at large x count more when deciding how to tilt the line.
- The ½ in J cancels the 2 that comes from differentiating the square — that is why we put it there.

> 💡 **Hint:** `np.mean(...)` computes an average, which is the same as (1/m)·Σ. `errors * x` multiplies the two arrays element by element.

In [ ]:
# TODO: complete the gradient function (three lines).
def compute_gradient(x, y, w, b):
    """Return (dj_dw, dj_db): the slopes of the cost J in the w and b directions."""
    errors = None   # predictions minus measured values (an array)
    dj_dw = None    # average of errors * x
    dj_db = None    # average of errors
    return dj_dw, dj_db

In [ ]:
# ✅ Self-check: run this cell to test your work above
gw0, gb0 = compute_gradient(x, y, 0.0, 0.0)
assert gw0 is not None and gb0 is not None, "compute_gradient returned None. Finish the three TODO lines above first."
assert abs(gb0 - (-35.818)) < 0.01,     f"At w = b = 0, dJ/db should be -35.82 MPa (minus the average strength), but yours is {gb0:.2f}. dj_db is the average of the errors, and errors = predictions - y."
assert abs(gw0 - (-24.177)) < 0.01,     f"At w = b = 0, dJ/dw should be -24.18, but yours is {gw0:.2f}. dj_dw is the average of errors * x (multiply by x before averaging)."
gw, gb = compute_gradient(x, y, w_best, b_best)
assert abs(gw) < 1e-6 and abs(gb) < 1e-6, "At the best line both slopes should be (almost) zero. Check the sign: errors = predict(...) - y."
print("Correct! At the bottom of the bowl both slopes are zero.")

**Check your formula a second way.** A good engineering habit: verify a result with an independent method.
The slope of a curve is about (rise ÷ run) over a tiny step h. If both methods agree, the formula is right.

In [ ]:
h = 1e-4   # a tiny step
w0, b0 = 0.0, 0.0
numeric_dw = (compute_cost(x, y, w0 + h, b0) - compute_cost(x, y, w0 - h, b0)) / (2 * h)
numeric_db = (compute_cost(x, y, w0, b0 + h) - compute_cost(x, y, w0, b0 - h)) / (2 * h)
dj_dw, dj_db = compute_gradient(x, y, w0, b0)
print(f"dJ/dw: formula = {dj_dw:.4f}   numerical = {numeric_dw:.4f}")
print(f"dJ/db: formula = {dj_db:.4f}   numerical = {numeric_db:.4f}")

**One step by hand.** The update rule (next part) is w ← w − α·∂J/∂w. At (0, 0) with α = 0.1:
w ← 0 − 0.1 · (−24.2) = 0 + 2.4 = **+2.4 MPa**. Subtracting a negative number makes w go *up*.

**✍️ Question:** Do the same for b: with α = 0.1, what is b after the first step? Is that the right direction for b? The best w is about −26.6 MPa — so is the first move of w (to +2.4) in the right direction?

*Your answer:* 

## Part 3 — The descent loop (15 min)
Repeat many times:

$$w \leftarrow w - \alpha\,\frac{\partial J}{\partial w} \qquad\qquad b \leftarrow b - \alpha\,\frac{\partial J}{\partial b}$$

- **α (alpha), the learning rate**, sets the step size.
- The minus sign means "go *against* the slope" — downhill.
- **Simultaneous update:** compute *both* slopes at the current (w, b) first, then change both. If you updated w
  first and then computed the b-slope with the new w, you would be mixing two different points.

`for it in range(num_iters):` repeats the indented block `num_iters` times. We also record the cost and the (w, b)
point after every step so we can plot them.

> 💡 **Hint:** Three lines: `dj_dw, dj_db = compute_gradient(x, y, w, b)`, then `w = w - alpha * dj_dw` and `b = b - alpha * dj_db`. Keep the indentation (4 extra spaces inside the loop).

In [ ]:
def gradient_descent(x, y, w0, b0, alpha, num_iters):
    """Run gradient descent from (w0, b0). Returns final w, b, the cost history, and the path of (w, b) points."""
    w, b = w0, b0
    J_history = [compute_cost(x, y, w, b)]   # cost before the first step
    path = [(w, b)]
    for it in range(num_iters):
        # TODO 1: compute both slopes at the current (w, b) with compute_gradient

        # TODO 2: update w and b (simultaneous update)

        J_history.append(compute_cost(x, y, w, b))
        path.append((w, b))
    return w, b, J_history, path

In [ ]:
w_gd, b_gd, J_hist, path = gradient_descent(x, y, w0=0.0, b0=0.0, alpha=0.1, num_iters=1000)
print(f"Gradient descent: w = {w_gd:.2f} MPa, b = {b_gd:.2f} MPa, J = {J_hist[-1]:.3f} MPa²")
print(f"np.polyfit:       w = {w_best:.2f} MPa, b = {b_best:.2f} MPa, J = {compute_cost(x, y, w_best, b_best):.3f} MPa²")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert not (w_gd == 0 and b_gd == 0),     "w and b are still 0: the update lines are missing, or they are not indented inside the for loop (4 more spaces than 'for')."
assert len(J_hist) == 1001, f"J_hist should hold 1001 costs (start + 1000 steps), but it has {len(J_hist)}. Keep the two append lines inside the loop."
assert abs(w_gd - (-26.55)) < 0.1 and abs(b_gd - 55.68) < 0.1,     f"Expected w ≈ -26.55 and b ≈ 55.68 after 1000 steps, got {w_gd:.2f} and {b_gd:.2f}. Check the minus signs (w - alpha * dj_dw) and that both slopes are computed BEFORE w or b changes."
print("Correct! Gradient descent found (almost) the same line as polyfit.")

A **learning curve** plots the cost after each step (iteration). It is the first thing to look at when
training any model: it should go down and level off.

> 💡 **Hint:** `plt.plot(J_hist)` uses 0, 1, 2, … as the x-values automatically. Then add `plt.xlabel(...)`, `plt.ylabel(...)`, `plt.title(...)`, and `plt.show()`.

In [ ]:
# TODO: plot the learning curve: J_hist (y-axis) vs iteration number (x-axis).
#       Add a title and axis labels with units.

Now the path of gradient descent on the cost contour map from Lab 03 (red dots = every 10th step).

In [ ]:
w_grid = np.linspace(-40, 20, 121)   # MPa
b_grid = np.linspace(-5, 80, 86)     # MPa
J_grid = np.zeros((len(b_grid), len(w_grid)))
for i, b in enumerate(b_grid):
    for j, w in enumerate(w_grid):
        J_grid[i, j] = compute_cost(x, y, w, b)
W, B = np.meshgrid(w_grid, b_grid)
levels = [105, 108, 115, 130, 160, 220, 350, 600, 1000, 1600]   # MPa²

def plot_path(path, title, every=10):
    """Draw the cost contours and a gradient-descent path."""
    pw = [p[0] for p in path][::every]
    pb = [p[1] for p in path][::every]
    plt.figure(figsize=(7, 5))
    cs = plt.contour(W, B, J_grid, levels=levels, cmap="viridis")
    plt.clabel(cs, fontsize=8, fmt="%.0f")
    plt.plot(pw, pb, "o-", color="red", markersize=3, linewidth=1, label="gradient-descent path")
    plt.scatter([path[0][0]], [path[0][1]], color="black", s=60, zorder=3, label="start")
    plt.scatter([w_best], [b_best], color="gold", edgecolor="black", marker="*", s=250, zorder=3, label="minimum (polyfit)")
    plt.xlabel("w (MPa)"); plt.ylabel("b (MPa)")
    plt.title(title)
    plt.legend(loc="lower left", fontsize=9)
    plt.show()

plot_path(path, "Gradient descent on the cost contours (α = 0.1)")

**✍️ Question:** Describe the path. Why does it bend instead of heading straight for the minimum? (Hint: at every point, the step goes straight downhill — perpendicular to the contour line — not toward the minimum.)

*Your answer:* 

## Part 4 — The learning rate α (12 min)
- **Too small:** tiny steps — the cost goes down, but painfully slowly.
- **Too large:** each step overshoots the bottom; the cost can even grow without limit (**divergence**).
- **Just right:** the cost drops quickly and levels off.

A **dictionary** (Lab 01) is a good place to store one result per α: `results[alpha] = ...`.

> 💡 **Hint:** `results[alpha] = gradient_descent(x, y, 0.0, 0.0, alpha, 1000)` — the four outputs are stored together. Delete `pass`.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1.0]
results = {}
for alpha in alphas:
    # TODO: run gradient descent from (0, 0) for 1000 iterations with this alpha,
    #       and store the four outputs in results[alpha]
    pass

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert len(results) > 0, "results is still empty. Inside the loop, store results[alpha] = gradient_descent(...) and delete pass."
assert len(results) == 4, f"results should have one entry per alpha (4), but it has {len(results)}."
print("Correct!")

`plt.yscale("log")` spaces the y-axis by factors of 10, so small and large costs both stay visible.

In [ ]:
plt.figure(figsize=(8, 4.5))
for alpha in alphas:
    w_a, b_a, J_a, path_a = results[alpha]
    plt.plot(J_a, label=f"α = {alpha}")
    print(f"α = {alpha:<6}  after 1000 steps: w = {w_a:7.2f} MPa, b = {b_a:6.2f} MPa, J = {J_a[-1]:7.2f} MPa²")
plt.axhline(compute_cost(x, y, w_best, b_best), color="black", linestyle="--", linewidth=1, label="minimum J")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost J (MPa², log scale)")
plt.title("Learning curves for four learning rates")
plt.legend()
plt.show()

Now a learning rate that is a little **too large**. We run only 30 steps — watch the cost. (The lecture's one-dimensional picture held b fixed, and there a larger α was still safe. Here w and b both move, and on this data the limit is about α ≈ 1.25.)

In [ ]:
w_big, b_big, J_big, path_big = gradient_descent(x, y, 0.0, 0.0, alpha=1.3, num_iters=30)
print("Cost after steps 0-5:", [round(J) for J in J_big[:6]], "MPa²")
print(f"Cost after 30 steps: {J_big[-1]:,.0f} MPa²")

plt.figure(figsize=(7, 4))
plt.plot(J_big, color="red")
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost J (MPa², log scale)")
plt.title("α = 1.3: the cost grows every step (divergence)")
plt.show()

**✍️ Question:** Which α was too small, which was slow but OK, which was fastest, and which diverged? Look at the printed w for α = 0.001: what is strange about it, and why would that be dangerous if you stopped there and used the model?

*Your answer:* 

## Part 5 — Feature scaling (8 min)
The valley was long and narrow partly because of the scale of x. **Standardizing** (z-score) a feature rescales
it to mean 0 and standard deviation 1:

$$x_{\text{scaled}} = \frac{x - \text{mean}(x)}{\text{std}(x)}$$

Each scaled value says "how many standard deviations above (+) or below (−) the average this mix is".

> 💡 **Hint:** Use `np.mean(x)` and `np.std(x)`. **Parentheses!** `x_scaled = (x - x_mean) / x_std`. Without them, Python divides first: `x - x_mean / x_std` is not the same thing.

In [ ]:
# TODO: standardize x.
x_mean = None
x_std = None
x_scaled = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert x_scaled is not None, "x_scaled is still None. Finish the TODO above first."
assert abs(np.mean(x_scaled)) < 1e-9 and abs(np.std(x_scaled) - 1) < 1e-9,     f"x_scaled should have mean 0 and std 1, but it has mean {np.mean(x_scaled):.2f} and std {np.std(x_scaled):.2f}. Did you write (x - x_mean) / x_std with the parentheses?"
print("Correct!")

In [ ]:
ws_s, bs_s, J_s, path_s = gradient_descent(x_scaled, y, 0.0, 0.0, alpha=0.1, num_iters=100)
J_raw_100 = results[0.1][2][:101]   # first 100 steps of the unscaled run with the same alpha

plt.figure(figsize=(7, 4))
plt.plot(J_raw_100, label="original x, α = 0.1")
plt.plot(J_s, label="scaled x, α = 0.1")
plt.yscale("log")
plt.xlabel("Iteration"); plt.ylabel("Cost J (MPa², log scale)")
plt.title("Same learning rate, 100 steps: scaled vs original feature")
plt.legend(); plt.show()
print(f"After 100 steps: original x J = {J_raw_100[-1]:.2f} MPa²,  scaled x J = {J_s[-1]:.2f} MPa²")

# Convert the scaled-feature line back to original units (algebra on w*(x - mean)/std + b)
w_back = ws_s / x_std
b_back = bs_s - ws_s * x_mean / x_std
print(f"Back in original units: w = {w_back:.2f} MPa, b = {b_back:.2f} MPa   (polyfit: {w_best:.2f}, {b_best:.2f})")

Scaling matters even more when a feature has **large numbers**. Cement is 102–540 kg/m³. Watch five steps
of gradient descent on *raw* cement content with a "small" α = 0.001:

In [ ]:
cement = df["cement_kg_m3"].to_numpy()   # kg/m³
_, _, J_cem, _ = gradient_descent(cement, y, 0.0, 0.0, alpha=0.001, num_iters=5)
print("Raw cement, α = 0.001 — cost after each step:", [f"{J:.1e}" for J in J_cem], "MPa²")

cement_scaled = (cement - cement.mean()) / cement.std()
_, _, J_cem_s, _ = gradient_descent(cement_scaled, y, 0.0, 0.0, alpha=0.1, num_iters=100)
print(f"Scaled cement, α = 0.1 — cost after 100 steps: {J_cem_s[-1]:.1f} MPa²")

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). (a) With the same α = 0.1, how did the scaled run compare with the original one after 100 steps? (b) Did scaling change the final line, once converted back to original units? (c) Why do you think the cost of the raw-cement run grew so fast, even with α = 0.001?

*Your answer:* 

## Part 6 — All 8 inputs with scikit-learn (20 min)
Strength depends on every ingredient and on age, not just w/c. **Multiple linear regression** uses all of them:

$$\hat{y} = w_1x_1 + w_2x_2 + \dots + w_8x_8 + b$$

Same recipe: squared-error cost, find the (w₁ … w₈, b) at the bottom. **scikit-learn** (`sklearn`) is the standard
Python library for this. Its `LinearRegression` finds the bottom directly with linear algebra (like `polyfit`),
so there is no α to tune; gradient descent is what you would use for very large data and for neural networks.

### 6.1 Train/test split
If we judge a model on the same mixes it learned from, it can look better than it really is. So we **hold back 20%
of the tests** as a **test set** that the model never sees during training — like checking a design against a load
case you did not use to size it. `random_state=42` fixes the random shuffle so everyone gets the same split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

feature_cols = ["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3",
                "superplasticizer_kg_m3", "coarse_agg_kg_m3", "fine_agg_kg_m3", "age_days"]
X = df[feature_cols]
y_all = df["strength_mpa"]

X_train, X_test, y_train, y_test = train_test_split(X, y_all, test_size=0.2, random_state=42)
print("Training set:", X_train.shape, "  Test set:", X_test.shape)

### 6.2 Scale, then fit
`StandardScaler` does the same z-score as Part 5, for every column. **Fit it on the training set only**
(`fit_transform` learns each column's mean and std, then applies them), and then apply those *same* numbers to the
test set with `transform`. The test set must stay unseen — even its mean and std.

> 💡 **Hint:** `scaler = StandardScaler()`, then `X_train_s = scaler.fit_transform(X_train)` and `X_test_s = scaler.transform(X_test)` (no `fit` for the test set!).

In [ ]:
# TODO: create a StandardScaler, fit it on X_train and transform X_train; then transform X_test.
scaler = None
X_train_s = None
X_test_s = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert X_train_s is not None and X_test_s is not None, "X_train_s or X_test_s is still None. Finish the TODO above first."
assert abs(X_train_s.mean()) < 1e-9, "The scaled training columns should have mean 0. Use scaler.fit_transform(X_train) for the training set."
assert X_test_s.shape == (206, 8), f"X_test_s should be 206 rows × 8 columns, but it is {X_test_s.shape}. Use scaler.transform(X_test)."
print("Correct!")

Every scikit-learn model works the same way: create it, `.fit(X, y)` to learn, `.predict(X)` to predict.

> 💡 **Hint:** `model = LinearRegression()`, `model.fit(X_train_s, y_train)`, then `model.predict(X_train_s)` and `model.predict(X_test_s)`.

In [ ]:
# TODO: create a LinearRegression model, fit it on the scaled training data,
#       and predict strengths for the training set and the test set.
model = None
y_pred_train = None
y_pred_test = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert model is not None and y_pred_train is not None and y_pred_test is not None, "Something is still None. Finish the TODO above first."
assert len(y_pred_train) == 824 and len(y_pred_test) == 206,     f"Expected 824 training and 206 test predictions, got {len(y_pred_train)} and {len(y_pred_test)}. Predict X_train_s for one and X_test_s for the other."
print("Correct!")

### 6.3 How good is it?
- **MAE** = average |ŷ − y|, in MPa — the typical size of an error.
- **RMSE** = $\sqrt{\text{average}(\hat{y} - y)^2}$, in MPa — like MAE, but punishes big errors more.
- **R²** = $1 - \dfrac{\sum(\hat{y} - y)^2}{\sum(y - \bar{y})^2}$ — share of the variation explained (1 = perfect,
  0 = no better than always predicting the average).

A **baseline** is the simplest possible model — here, one that always predicts the average strength of the training
mixes, whatever the inputs. Every useful model must beat its baseline; if it does not, it has learned nothing useful.
We compare three models on the **same test set**: the baseline, the one-feature w/c line, and the 8-input model.

In [ ]:
def metrics(y_true, y_pred):
    """Return MAE (MPa), RMSE (MPa), and R²."""
    return (mean_absolute_error(y_true, y_pred),
            np.sqrt(mean_squared_error(y_true, y_pred)),
            r2_score(y_true, y_pred))

# Baseline: always predict the average strength of the training set
baseline_pred = np.full(len(y_test), y_train.mean())

# One-feature model: the w/c line, fitted on the training set only
wc_train = (X_train["water_kg_m3"] / X_train["cement_kg_m3"]).to_numpy()
wc_test = (X_test["water_kg_m3"] / X_test["cement_kg_m3"]).to_numpy()
w1, b1 = np.polyfit(wc_train, y_train, 1)
wc_pred = predict(wc_test, w1, b1)

rows = {
    "Baseline (average), test": metrics(y_test, baseline_pred),
    "w/c line, test": metrics(y_test, wc_pred),
    "8 inputs, train": metrics(y_train, y_pred_train),
    "8 inputs, test": metrics(y_test, y_pred_test),
}
pd.DataFrame(rows, index=["MAE (MPa)", "RMSE (MPa)", "R²"]).T.round(2)

**✍️ Question:** (a) Compare the 8-input model's train and test numbers: is there a sign of overfitting? (b) How much better is it than the baseline and the w/c line? (c) Is a typical error of this size good enough to accept a structural mix without testing cylinders?

*Your answer:* 

> 💡 **Hint:** `plt.scatter(y_test, y_pred_test, alpha=0.6)`, then `plt.plot([0, 85], [0, 85], "k--", label="perfect prediction")` draws the 45° line. Finish with labels, `plt.legend()`, `plt.show()`.

In [ ]:
# TODO: scatter plot of actual strength (x-axis) vs predicted strength (y-axis) for the TEST set.
#       Add the dashed line where predicted = actual, a legend, a title, and axis labels with units.

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). On your plot, where does the model do worst? Look at the weakest and the strongest mixes. Are they predicted too high or too low?

*Your answer:* 

### 6.4 What did the model learn? Coefficients — interpret with care
Because the inputs were standardized, each coefficient is **the change in predicted strength (MPa) when that input
rises by one standard deviation while all the other inputs stay the same**. The intercept is the prediction for an
"average" mix (every input at its training mean).

In [ ]:
coefs = pd.Series(model.coef_, index=feature_cols).sort_values()
print(f"Intercept: {model.intercept_:.2f} MPa")
print("One standard deviation of each input in the training set:")
print(pd.Series(scaler.scale_, index=feature_cols).round(1).to_string())

plt.figure(figsize=(7, 4))
plt.barh(coefs.index, coefs.values, color=["#AB0520" if v < 0 else "#1E5288" for v in coefs.values])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Change in predicted strength per +1 std of the input (MPa)")
plt.title("Coefficients of the 8-input model (standardized inputs)")
plt.tight_layout()
plt.show()

**Before you draw conclusions:**
- *"All other inputs stay the same"* is often impossible in a real mix: 1 m³ of concrete is full, so adding cement
  means less of something else.
- These are **associations in this dataset**, not proven causes. Ingredients that tend to appear together share
  credit in ways that can mislead (look at the aggregates).
- The model is a straight line in every input, but the age effect is curved (fast early gain, then leveling off).
- It is only trustworthy **inside the range of the data** (e.g., cement 102–540 kg/m³, age 1–365 days).
- **Redundant inputs (collinearity).** If one input can be computed exactly from others (for example, a "total
  binder" column = cement + slag + fly ash), the model cannot tell their effects apart: many different sets of
  coefficients give exactly the same predictions, so the individual coefficients mean little. Inputs that are only
  strongly related cause a milder version of the same problem. (You will meet a real case in HW2.)

**✍️ Question:** 🏠 **Finish at home** (still part of the lab, due Sunday). A classmate says: "Cement has the biggest coefficient (≈12.8), so adding 100 kg/m³ of cement always adds about 12.8 MPa." Give two problems with this statement.

*Your answer:* 

---
## 🎟️ Exit ticket
Three short questions you answer **alone, without AI (AI-Free)**, during the last 10 minutes of lab — even if you have not finished every part above. Use your own words, 1–3 sentences each. (If you must leave early, answer them on your own before you submit.)

**1.** In one sentence, what does the gradient tell gradient descent to do?

*Your answer:* 

**2.** What did you see when the learning rate α was too large, and when it was too small?

*Your answer:* 

**3.** Why do we report MAE, RMSE, and R² on a test set that the model never saw during training?

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
**Due:** Sunday 11:59 pm (Arizona time) of this week, on Brightspace. Late work: see the syllabus late-work table (two grace days per semester, then −10% per day, up to 3 days).

1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.